In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# --- Data Loading and Cleaning ---
df = pd.read_csv('dataset_mansion.csv')

df['Consumption'] = pd.to_numeric(
    df['Consumption'].astype('string').str.replace(',', '.', regex=False),
    errors='raise'
)

df['MTime'] = pd.to_datetime(df['MTime'], errors='raise', utc=True)
assert df['MTime'].notna().all(), 'Check missing timestamps first.'
assert not df['MTime'].duplicated().any(), 'Check duplicate timestamps first.'
df = df.set_index('MTime').sort_index()

# --- Feature Removal ---
columns_to_drop = [
    'Latitude', 'Longitude', 'Place', 'PresentWeather(rank)',
    'HighestTemperature(degC)', 'LowestTemperature(degC)',
    'MaximumWindSpeed(m/s)', 'MinimumWindSpeed(m/s)',
    'MaximumPrecipitationIntensity(mm/h)'
]
df = df.drop(columns=columns_to_drop)

# --- Seasonal Imputation ---
df = df.asfreq('h')

# 1. Safely interpolate ONLY small gaps (e.g., up to 4 hours)
df = df.interpolate(method='time', limit=4)
# 2. For the 20 to 58-hour gaps, fill using the exact same hour the day before (up to 72 hrs)
for _ in range(3): 
    df = df.fillna(df.shift(24))
# 3. If any gaps survive, fill with the historical average for that specific month and hour
df = df.fillna(df.groupby([df.index.month, df.index.hour]).transform('mean'))
# 4. Final safety net
df = df.bfill().ffill()

# --- Feature Engineering ---
df_features = df.copy()
local_time = df_features.index.tz_convert('Europe/Helsinki')
hour = local_time.hour
month = local_time.month - 1
day_of_week = local_time.dayofweek

df_features['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df_features['hour_cos'] = np.cos(2 * np.pi * hour / 24)
df_features['day_sin'] = np.sin(2 * np.pi * day_of_week / 7)
df_features['day_cos'] = np.cos(2 * np.pi * day_of_week / 7)
df_features['month_sin'] = np.sin(2 * np.pi * month / 12)
df_features['month_cos'] = np.cos(2 * np.pi * month / 12)

# --- Wind Direction Transformation ---
# 1. Calculate the cyclical nature of the wind
wind_radians = df_features['WindDirection(deg)'] * (np.pi / 180)
df_features['wind_dir_sin'] = np.sin(wind_radians)
df_features['wind_dir_cos'] = np.cos(wind_radians)

# ßDROP the raw wind direction column now that we have the clean sin/cos versions
df_features = df_features.drop(columns=['WindDirection(deg)'])


df_features['is_weekend'] = (local_time.dayofweek >= 5).astype(int)
df_features['is_covid'] = (local_time >= '2020-03-01').astype(int)

# Holidays
week_of_year = local_time.isocalendar().week
local_month = local_time.month
day = local_time.day
is_summer_holiday = ((local_month == 6) & (day >= 8)) | (local_month == 7) | ((local_month == 8) & (day <= 7))
df_features['is_holiday'] = (is_summer_holiday | (week_of_year == 9) | (week_of_year == 42) | ((local_month == 12) & (day >= 24) & (day <= 26))).astype(int)

# Working hours & routines
df_features['is_working_hour'] = ((hour >= 8) & (hour < 16) & (df_features['is_weekend'] == 0) & (df_features['is_holiday'] == 0)).astype(int)
df_features['is_evening_routine'] = ((hour >= 17) & (hour <= 23)).astype(int)

# --- Splitting & Scaling ---
train_df = df_features.loc['2016-01-01':'2020-09-30'].copy()
test_df = df_features.loc['2020-10-01':'2021-09-30'].copy()
eval1_df = df_features.loc['2021-10-01':'2021-12-31'].copy()
eval2_df = df_features.loc['2022-01-01':'2022-02-28'].copy()
eval3_df = df_features.loc['2022-03-01':'2022-05-31'].copy()

continuous_features = ['AirTemperature(degC)', 'RelativeHumidity(%)', 'WindSpeed(m/s)', 'PrecipitationAmount(mm)', 'AirPressure(hPa)']
scaler = StandardScaler()

train_df[continuous_features] = scaler.fit_transform(train_df[continuous_features])
test_df[continuous_features] = scaler.transform(test_df[continuous_features])
eval1_df[continuous_features] = scaler.transform(eval1_df[continuous_features])
eval2_df[continuous_features] = scaler.transform(eval2_df[continuous_features])
eval3_df[continuous_features] = scaler.transform(eval3_df[continuous_features])

print("Preprocessing complete. Data is ready for the TCN.")

Preprocessing complete. Data is ready for the TCN.


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Define the TimeSeriesDataset class
class TimeSeriesDataset(Dataset):
    def __init__(self, df, target_col_name, seq_len=48, pred_len=1):
        self.data = df.values.astype(np.float32)
        self.target_idx = df.columns.get_loc(target_col_name)
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, index):
        # X: Past 'seq_len' hours of all features
        x = self.data[index : index + self.seq_len]
        # Y: The next 1 hour of Consumption[cite: 6]
        y = self.data[index + self.seq_len : index + self.seq_len + self.pred_len, self.target_idx]
        return torch.tensor(x), torch.tensor(y)

SEQ_LEN = 48  # Look back 48 hours to predict the next hour
BATCH_SIZE = 64

# Create Datasets using your split dataframes
train_dataset = TimeSeriesDataset(train_df, 'Consumption', SEQ_LEN)
test_dataset = TimeSeriesDataset(test_df, 'Consumption', SEQ_LEN)
eval1_dataset = TimeSeriesDataset(eval1_df, 'Consumption', SEQ_LEN)
eval2_dataset = TimeSeriesDataset(eval2_df, 'Consumption', SEQ_LEN)
eval3_dataset = TimeSeriesDataset(eval3_df, 'Consumption', SEQ_LEN)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
eval1_loader = DataLoader(eval1_dataset, batch_size=BATCH_SIZE, shuffle=False)
eval2_loader = DataLoader(eval2_dataset, batch_size=BATCH_SIZE, shuffle=False)
eval3_loader = DataLoader(eval3_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [10]:
# Define the SimpleTCN model

class SimpleTCN(nn.Module):
    def __init__(self, num_features, num_channels, kernel_size=3):
        super(SimpleTCN, self).__init__()
        layers = []
        in_channels = num_features
        
        # Build 1D Convolutional layers with increasing dilations (1, 2, 4, 8)
        for out_channels in num_channels:
            dilation = len(layers) // 2 + 1 
            layers += [
                nn.Conv1d(in_channels, out_channels, kernel_size, padding=(kernel_size-1)*dilation, dilation=dilation),
                nn.ReLU(),
                nn.Dropout(0.2)
            ]
            in_channels = out_channels
            
        self.tcn = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], 1) # Output next hour prediction

    def forward(self, x):
        # PyTorch Conv1d expects shape: [Batch, Features, Sequence_Length]
        x = x.permute(0, 2, 1) 
        
        y = self.tcn(x)
        # Take the output of the last time step
        y = y[:, :, -1] 
        return self.linear(y)

# Initialize Model
NUM_FEATURES = train_df.shape[1]
model = SimpleTCN(num_features=NUM_FEATURES, num_channels=[32, 32, 64, 64])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.MSELoss() # Assignment requires appropriate loss function[cite: 6]
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- TRAINING LOOP ---
epochs = 20
print("Starting Training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        predictions = model(x_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Train MSE Loss: {train_loss/len(train_loader):.4f}")

Starting Training...
Epoch 1/20 | Train MSE Loss: 4.8359
Epoch 2/20 | Train MSE Loss: 2.2872
Epoch 3/20 | Train MSE Loss: 1.8817
Epoch 4/20 | Train MSE Loss: 1.8127
Epoch 5/20 | Train MSE Loss: 1.7891
Epoch 6/20 | Train MSE Loss: 1.7658
Epoch 7/20 | Train MSE Loss: 1.7428
Epoch 8/20 | Train MSE Loss: 1.7388
Epoch 9/20 | Train MSE Loss: 1.7325
Epoch 10/20 | Train MSE Loss: 1.7277
Epoch 11/20 | Train MSE Loss: 1.7192
Epoch 12/20 | Train MSE Loss: 1.7109
Epoch 13/20 | Train MSE Loss: 1.7040
Epoch 14/20 | Train MSE Loss: 1.7016
Epoch 15/20 | Train MSE Loss: 1.6962
Epoch 16/20 | Train MSE Loss: 1.7041
Epoch 17/20 | Train MSE Loss: 1.7031
Epoch 18/20 | Train MSE Loss: 1.6920
Epoch 19/20 | Train MSE Loss: 1.6919
Epoch 20/20 | Train MSE Loss: 1.6941


In [11]:
import math

# Define a function to evaluate the model on a given DataLoader
def evaluate_model(loader, dataset_name):
    model.eval()
    mse_loss = 0.0
    mae_loss = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            predictions = model(x_batch)
            
            mse_loss += nn.MSELoss(reduction='sum')(predictions, y_batch).item()
            mae_loss += nn.L1Loss(reduction='sum')(predictions, y_batch).item()
            total_samples += y_batch.size(0)
            
    mse = mse_loss / total_samples
    rmse = math.sqrt(mse)
    mae = mae_loss / total_samples
    return rmse, mae

# Gather results
results = {
    "Dataset": ["Training", "Test (2020-2021)", "Eval 1 (Late 2021)", "Eval 2 (Early 2022)", "Eval 3 (Spring 2022)"],
    "RMSE (kWh Error)": [
        evaluate_model(train_loader, "Train")[0],
        evaluate_model(test_loader, "Test")[0],
        evaluate_model(eval1_loader, "Eval 1")[0],
        evaluate_model(eval2_loader, "Eval 2")[0],
        evaluate_model(eval3_loader, "Eval 3")[0]
    ],
    "MAE (kWh Error)": [
        evaluate_model(train_loader, "Train")[1],
        evaluate_model(test_loader, "Test")[1],
        evaluate_model(eval1_loader, "Eval 1")[1],
        evaluate_model(eval2_loader, "Eval 2")[1],
        evaluate_model(eval3_loader, "Eval 3")[1]
    ]
}

# Display as a clean table
results_df = pd.DataFrame(results)
print("\n--- Final Model Accuracy Table ---")
print(results_df.round(3).to_string(index=False))


--- Final Model Accuracy Table ---
             Dataset  RMSE (kWh Error)  MAE (kWh Error)
            Training             1.289            1.044
    Test (2020-2021)             1.355            1.082
  Eval 1 (Late 2021)             1.662            1.282
 Eval 2 (Early 2022)             1.946            1.610
Eval 3 (Spring 2022)             0.999            0.829
